# Train Commutative CNN

This notebook intentionally keeps orchestration thin. Split preparation, fitting, reporting, plotting, and persistence are delegated to shared utilities in `src.ml`.


In [ ]:
%load_ext autoreload
%autoreload 2

from dataclasses import asdict
from pathlib import Path

import pandas as pd

from src.ml import (
    LossWeightConfig,
    OptimizationConfig,
    display_experiment_summary,
    display_holdout_evaluation,
    fit_estimator_on_experiment,
    persist_experiment_artifacts,
    plot_holdout_branch_embedding_projections,
    plot_training_history,
    prepare_multitask_experiment_data,
)
from src.dataset_config import load_current_dataset_artifact_path
from src.tensor_utils import build_tensor_embedding_2d, load_labeled_tensor_dataset, plot_tensor_embedding_2d

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.expand_frame_repr", False)


In [ ]:
from src.ml import CommutativeCNNClassifier, CommutativeCNNConfig


In [ ]:
# User inputs

dataset_artifact_path = load_current_dataset_artifact_path()
print(dataset_artifact_path.name)
experiment_output_dir = Path("artifacts/nb7_commutative_cnn")
persist_artifacts = True

holdout_fraction = 0.25
validation_fraction_within_train = 0.20
train_num_random_rotations = 6
rotation_range_degrees = 12.0

model_config = CommutativeCNNConfig(
    spatial_conv_channels=(6, 8),
    spatial_kernel_size_z=(1, 1),
    spatial_kernel_size_xy=(5, 3),
    spatial_stride_z=(1, 1),
    spatial_stride_xy=(1, 1),
    spatial_pool_kernel_z=(1, 1),
    spatial_pool_kernel_xy=(2, 2),
    spatial_pool_stride_z=(1, 1),
    spatial_pool_stride_xy=(2, 2),
    temporal_st_channels=(12,),
    temporal_st_kernel_sizes=(3,),
    temporal_ts_channels=(8,),
    temporal_ts_kernel_sizes=(5,),
    spatial_agg_channels=(12,),
    spatial_agg_kernel_size_z=(3,),
    spatial_agg_kernel_size_xy=(3,),
    spatial_agg_stride_z=(1,),
    spatial_agg_stride_xy=(1,),
    spatial_agg_pool_kernel_z=(1,),
    spatial_agg_pool_kernel_xy=(2,),
    spatial_agg_pool_stride_z=(1,),
    spatial_agg_pool_stride_xy=(2,),
    patch_size_z=1,
    patch_size_xy=16,
    embedding_dim=8,
    probe_local_count=32,
    probe_region_grid=(1, 2, 2),
    probe_time_bins=8,
    probe_frequency_bins=4,
    dropout=0.5,
)
optimization_config = OptimizationConfig(
    batch_size=8,
    epochs=100,
    learning_rate=2e-4,
    weight_decay=3e-3,
    early_stopping_patience=4,
    early_stopping_min_delta=0.0,
    scheduler_patience=1,
    scheduler_factor=0.5,
    scheduler_min_lr=1e-6,
    validation_split=0.0,
    random_state=0,
    standardize=True,
    device=None,
    verbose=True,
)
loss_weight_config = LossWeightConfig(
    action_weight=1.0,
    compound_weight=0.05,
    concentration_weight=0.05,
    lambda_align=0.0,
)


In [ ]:
dataset = load_labeled_tensor_dataset(dataset_artifact_path)


In [ ]:
experiment = prepare_multitask_experiment_data(
    dataset,
    holdout_fraction=holdout_fraction,
    validation_fraction_within_train=validation_fraction_within_train,
    train_num_random_rotations=train_num_random_rotations,
    rotation_range_degrees=rotation_range_degrees,
    random_state=optimization_config.random_state,
)


In [ ]:
display_experiment_summary(experiment)


In [ ]:
model = CommutativeCNNClassifier(
    model_config=model_config,
    optimization_config=optimization_config,
    loss_weight_config=loss_weight_config,
)


In [ ]:
fit_estimator_on_experiment(model, experiment)


In [ ]:
plot_training_history(model, title="Commutative CNN loss curves", loess_frac=0.6);


In [ ]:
holdout_evaluation = display_holdout_evaluation(model, experiment)


In [ ]:
holdout_embedding_projection = build_tensor_embedding_2d(
    model.transform(experiment.splits.X_holdout),
    experiment.y_true_holdout["action"],
    label_map=experiment.label_maps["action"],
    metadata=experiment.splits.metadata_holdout,
    method="umap",
    random_state=optimization_config.random_state,
)
plot_tensor_embedding_2d(
    holdout_embedding_projection,
    title="Holdout embedding projection by action",
    marker_column="compound",
)


In [ ]:
run_config = {
    "dataset_artifact_path": dataset_artifact_path,
    "holdout_fraction": holdout_fraction,
    "validation_fraction_within_train": validation_fraction_within_train,
    "train_num_random_rotations": train_num_random_rotations,
    "rotation_range_degrees": rotation_range_degrees,
    "model_config": asdict(model_config),
    "optimization_config": asdict(optimization_config),
    "loss_weight_config": asdict(loss_weight_config),
}


In [ ]:
if persist_artifacts:
    experiment_artifacts = persist_experiment_artifacts(
        output_dir=experiment_output_dir,
        estimator=model,
        reports=holdout_evaluation.reports,
        config=run_config,
    )
